# Text-to-SQL Transformer — Colab + Google Drive

This notebook:
1. Mounts Google Drive
2. Trains from the project folder on Drive
3. Saves checkpoints to Drive
4. **Auto-resumes** if the session disconnects (run the train cell again)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Set your Drive project path
Change this if your folder name is different.

In [ ]:
import os

# EDIT THIS PATH to match your Google Drive folder
PROJECT_DIR = '/content/drive/MyDrive/GenAI_Project2/Gen AI Project 2'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')

os.makedirs(RESULTS_DIR, exist_ok=True)

%cd "{PROJECT_DIR}"
!ls
print('Results will be saved to:', RESULTS_DIR)

In [ ]:
!pip install -q -r requirements.txt

## Prepare data (run once; skip next times if already done on Drive)

In [ ]:
import os

need_data = not os.path.exists('WikiSQL/data/train.jsonl')
need_pairs = not os.path.exists('starter/processed/train_pairs.jsonl')
need_tok = not os.path.exists('starter/tokenizer_files/spm.model')

if need_data:
    !python download_wikisql.py
else:
    print('WikiSQL data already on Drive')

if need_pairs:
    !python starter/data_prep.py
else:
    print('pairs already prepared')

if need_tok:
    !python starter/tokenizer.py
else:
    print('tokenizer already ready')

## Train (safe to re-run after disconnect)

- Saves `results/last_model.pt` after every epoch
- Saves `results/best_model.pt` when dev loss improves
- If `last_model.pt` exists, training **continues** automatically

In [ ]:
# If session breaks: just run THIS cell again.
# It will resume from results/last_model.pt on Drive.

!python train.py --epochs 20 --batch_size 64 --out_dir "{RESULTS_DIR}" --resume

## Optional: start over from scratch

In [ ]:
# ONLY run this if you really want to delete progress and restart:
# !python train.py --epochs 20 --batch_size 64 --out_dir "{RESULTS_DIR}" --no_resume

## Check what was saved

In [ ]:
import os, torch

last_path = os.path.join(RESULTS_DIR, 'last_model.pt')
best_path = os.path.join(RESULTS_DIR, 'best_model.pt')

print('last exists:', os.path.exists(last_path), last_path)
print('best exists:', os.path.exists(best_path), best_path)

if os.path.exists(last_path):
    ckpt = torch.load(last_path, map_location='cpu')
    print('last finished epoch:', ckpt.get('epoch'))
    print('next epoch to run :', ckpt.get('next_epoch'))
    print('best_dev so far   :', ckpt.get('best_dev'))
    print('optimizer step    :', ckpt.get('step'))